In [12]:
# ============================================================
# Fix PEFT / torchao compatibility
# ============================================================

!pip uninstall -y torchao

In [13]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# ============================================================
# Project Root
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM"
)

assert PROJECT_ROOT.exists(), "Project directory not found."

print(PROJECT_ROOT)

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM


In [15]:
# ============================================================
# Add Project to Python Path
# ============================================================

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project added to Python Path.")

Project added to Python Path.


In [16]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

TEST_SPLIT = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.csv"
)

DPO_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "dpo_qwen"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "dpo"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0

print("Test dataset :", TEST_SPLIT.exists())
print("DPO model    :", DPO_MODEL_PATH.exists())
print("Output dir   :", OUTPUT_DIR)

Test dataset : False
DPO model    : True
Output dir   : /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/results/dpo


In [17]:
# ============================================================
# Imports
# ============================================================

import json
import pandas as pd

from tqdm.auto import tqdm

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

# ============================================================
# Load Prompt Template
# ============================================================

P01_PROMPT = (
    PROJECT_ROOT
    / "prompts"
    / "P01_Zero_Shot_Closed_Coding.txt"
).read_text(
    encoding="utf-8"
)

print(P01_PROMPT[:800])
from utils.inference import (
    predict_quote,
    parse_response,
)

You are a software engineering researcher specializing in qualitative analysis of psychological safety in software development teams.

Your task is to perform qualitative coding of the quotes provided below by identifying and classifying psychological safety challenges described in each quote.

The goal is to identify challenges related to psychological safety, such as interpersonal risk-taking, learning anxiety, defensive behaviors, or resistance to change.

---

Context for Classification

Psychological safety refers to a person's perception that they can take interpersonal risks in a team environment without fear of negative consequences such as embarrassment, rejection, or punishment.

---

Unit of Analysis

Each quote represents a single unit of analysis.

Even if multiple psychologic


In [18]:
# ============================================================
# Load Test Dataset
# ============================================================

from utils.dataset import (
    load_data,
    merge_data,
    split_data,
)

quotes, gold = load_data(PROJECT_ROOT)

dataset = merge_data(
    quotes,
    gold,
)

_, _, test_df = split_data(
    dataset,
)

print(f"Test samples: {len(test_df)}")

test_df.head()

Test samples: 12


,quote_id,quote_text,gold_category
0,Quote 71,Should the developer shoot down the idea becau...,Disagreeing with suggestions/ideas
1,Quote 57,I am on the product side and have an agile tea...,Expressing concerns
2,Quote 84,"I have a strange situation at work, where a co...",Seeking help
3,Quote 110,To do this I have discussion with related team...,Disagreeing with suggestions/ideas
4,Quote 27,How to influence a scrum team to do something ...,Recommending changes


In [19]:
# ============================================================
# Load Base Model
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Base model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded successfully.


In [20]:
# ============================================================
# Load DPO Adapter
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    DPO_MODEL_PATH,
)

model.eval()

print("DPO adapter loaded successfully.")

DPO adapter loaded successfully.


In [24]:
# ============================================================
# Run Inference
# ============================================================

predictions = []

for _, row in test_df.iterrows():

    response = predict_quote(
        model=model,
        tokenizer=tokenizer,
        prompt_template=P01_PROMPT,
        quote_id=row["quote_id"],
        quote_text=row["quote_text"],
    )

    try:

        parsed = parse_response(response)

        predicted_category = parsed.get(
            "category",
            None,
        )

    except Exception:

        print(f"\nFailed on {row['quote_id']}")
        print(response)
        print("-" * 80)

        predicted_category = None

    predictions.append(
        {
            "quote_id": row["quote_id"],
            "gold_category": row["gold_category"],
            "predicted_category": predicted_category,
            "raw_response": response,
        }
    )

print(f"\nFinished inference on {len(predictions)} quotes.")


Finished inference on 12 quotes.


In [25]:
# ============================================================
# Reload Inference Module
# ============================================================

import importlib
import utils.inference

importlib.reload(utils.inference)

from utils.inference import (
    predict_quote,
    parse_response,
)

print("Inference module reloaded.")

Inference module reloaded.


In [26]:
# ============================================================
# Build Results DataFrame
# ============================================================

results_df = pd.DataFrame(
    predictions
)

print(results_df)

print()

print(results_df["predicted_category"])

     quote_id                       gold_category  \
0    Quote 71  Disagreeing with suggestions/ideas   
1    Quote 57                 Expressing concerns   
2    Quote 84                        Seeking help   
3   Quote 110  Disagreeing with suggestions/ideas   
4    Quote 27                Recommending changes   
5    Quote 12  Disagreeing with suggestions/ideas   
6    Quote 54                 Expressing concerns   
7    Quote 05                Recommending changes   
8    Quote 89                Recommending changes   
9    Quote 48  Disagreeing with suggestions/ideas   
10   Quote 43         Drawing attention to errors   
11   Quote 95                 Expressing concerns   

             predicted_category  \
0   Drawing attention to errors   
1           Expressing concerns   
2           Expressing concerns   
3           Expressing concerns   
4           Expressing concerns   
5   Drawing attention to errors   
6           Expressing concerns   
7           Expressing concern

In [27]:
# ============================================================
# Remove Failed Predictions
# ============================================================

results_df = results_df.dropna(
    subset=["predicted_category"]
).reset_index(drop=True)

print(results_df)

     quote_id                       gold_category  \
0    Quote 71  Disagreeing with suggestions/ideas   
1    Quote 57                 Expressing concerns   
2    Quote 84                        Seeking help   
3   Quote 110  Disagreeing with suggestions/ideas   
4    Quote 27                Recommending changes   
5    Quote 12  Disagreeing with suggestions/ideas   
6    Quote 54                 Expressing concerns   
7    Quote 05                Recommending changes   
8    Quote 89                Recommending changes   
9    Quote 48  Disagreeing with suggestions/ideas   
10   Quote 43         Drawing attention to errors   
11   Quote 95                 Expressing concerns   

             predicted_category  \
0   Drawing attention to errors   
1           Expressing concerns   
2           Expressing concerns   
3           Expressing concerns   
4           Expressing concerns   
5   Drawing attention to errors   
6           Expressing concerns   
7           Expressing concern

In [28]:
# ============================================================
# Classification Metrics
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

evaluation_df = results_df.dropna(
    subset=["predicted_category"]
).reset_index(drop=True)

print(f"Evaluated samples: {len(evaluation_df)}")
print()

accuracy = accuracy_score(
    evaluation_df["gold_category"],
    evaluation_df["predicted_category"],
)

print(f"Accuracy: {accuracy:.4f}")
print()

print(
    classification_report(
        evaluation_df["gold_category"],
        evaluation_df["predicted_category"],
        zero_division=0,
    )
)

cm = confusion_matrix(
    evaluation_df["gold_category"],
    evaluation_df["predicted_category"],
)

print("Confusion Matrix:")
print(cm)

Evaluated samples: 12

Accuracy: 0.2500

                                    precision    recall  f1-score   support

Disagreeing with suggestions/ideas       0.00      0.00      0.00         4
       Drawing attention to errors       0.00      0.00      0.00         1
               Expressing concerns       0.33      1.00      0.50         3
              Recommending changes       0.00      0.00      0.00         3
                      Seeking help       0.00      0.00      0.00         1

                          accuracy                           0.25        12
                         macro avg       0.07      0.20      0.10        12
                      weighted avg       0.08      0.25      0.12        12

Confusion Matrix:
[[0 3 1 0 0]
 [0 0 1 0 0]
 [0 0 3 0 0]
 [0 0 3 0 0]
 [0 0 1 0 0]]


In [29]:
# ============================================================
# Save Evaluation Results
# ============================================================

results_path = OUTPUT_DIR / "dpo_predictions.csv"

results_df.to_csv(
    results_path,
    index=False,
)

print(f"Results saved to:\n{results_path}")

Results saved to:
/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/results/dpo/dpo_predictions.csv
